In [1]:
%pip install pandas numpy scikit-learn imbalanced-learn matplotlib seaborn

Defaulting to user installation because normal site-packages is not writeable
  Using cached pandas-3.1.1-cp313-cp313-win_amd64.whl.metadata (16 kB)
  Using cached numpy-2.5.4-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached scikit_learn-1.9.1-cp313-cp313-win_amd64.whl.metadata (9.3 kB)
  Using cached imbalanced_learn-0.14.2-py3-none-any.whl.metadata (8.9 kB)
  Using cached matplotlib-3.11.2-cp313-cp313-win_amd64.whl.metadata (80 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached scipy-1.18.1-cp313-cp313-win_amd64.whl.metadata (61 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached narwhals-2.28.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached sklearn_compat-0.1.6-py3-none-any.whl.metadata (22 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached contourpy-1.4.0-cp313-cp313-win_amd64.whl.metadata (3.8 kB)
  Usin


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd

df = pd.read_csv('../data/PS_20174392719_1491204439457_log.csv')

df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [2]:
print(df.shape)                              # rows and columns
print(df.dtypes)                             # column types
print(df.isnull().sum())                     # missing values
print("duplicates:", df.duplicated().sum())  # duplicated rows
print(df["isFraud"].value_counts())          # fraud vs normal
print(df["isFraud"].mean() * 100)            # fraud percentage
print(df["type"].value_counts())             # transaction types
print(df.groupby("type")["isFraud"].sum()) 

(6362620, 11)
step                int64
type                  str
amount            float64
nameOrig              str
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest              str
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64
duplicates: 0
isFraud
0    6354407
1       8213
Name: count, dtype: int64
0.12908204481801522
type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64
type
CASH_IN        0
CASH_OUT    4116
DEBIT          0
PAYMENT        0
TRANSFER    4097
Name: isFraud, dtype: int64


In [3]:
print("duplicates:", df.duplicated().sum())

duplicates: 0


In [4]:
df = df[df["type"].isin(["TRANSFER", "CASH_OUT"])].copy()
print(df.shape)
print(df["isFraud"].value_counts())

(2770409, 11)
isFraud
0    2762196
1       8213
Name: count, dtype: int64


In [5]:
df = df.drop(columns=[
    "nameOrig", "nameDest",                 # account IDs, not features
    "isFlaggedFraud",                       # only 16 cases
    "oldbalanceOrg", "newbalanceOrig",      # balance columns leak the label
    "oldbalanceDest", "newbalanceDest",
])
print(df.columns.tolist())
print(df.shape)

['step', 'type', 'amount', 'isFraud']
(2770409, 4)


In [6]:
df = pd.get_dummies(df, columns=["type"], drop_first=True, dtype=int)
print(df.columns.tolist())
df.head()

['step', 'amount', 'isFraud', 'type_TRANSFER']


,step,amount,isFraud,type_TRANSFER
2,1,181.00,1,1
3,1,181.00,1,0
15,1,229133.94,0,0
19,1,215310.30,0,1
24,1,311685.89,0,1


In [7]:
import numpy as np

df["hour"] = df["step"] % 24               # hour of day
df["day"] = df["step"] // 24               # day of the simulation
df["log_amount"] = np.log1p(df["amount"])  # reduces the effect of huge amounts
print(df.columns.tolist())
df.head()

['step', 'amount', 'isFraud', 'type_TRANSFER', 'hour', 'day', 'log_amount']


,step,amount,isFraud,type_TRANSFER,hour,day,log_amount
2,1,181.00,1,1,1,0,5.204007
3,1,181.00,1,0,1,0,5.204007
15,1,229133.94,0,0,1,0,12.342066
19,1,215310.30,0,1,1,0,12.279840
24,1,311685.89,0,1,1,0,12.649754


In [8]:
df.to_csv("../data/processed_data.csv", index=False)
print("saved", df.shape)

saved (2770409, 7)
